# ساخت checkpointهای تمام‌لایهٔ Qwen MoE (Top-2 / 32 experts)

این Notebook برای Google Colab آماده شده است. خودش خودکار اجرا نمی‌شود؛ Runtime را به GPU رایگانِ در دسترس تنظیم کنید و سلول‌ها را به‌ترتیب اجرا کنید. هیچ سرویس پولی فعال نمی‌شود.

خروجی این مرحله checkpoint جداگانه برای FFN هر لایه است، نه هنوز یک مدل کامل قابل بارگذاری در Transformers. فایل‌ها در Google Drive ذخیره می‌شوند تا با قطع اتصال Colab باقی بمانند. اجرای ۲۴ لایه ممکن است زمان‌بر باشد.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
WORK = Path('/content/drive/MyDrive/qwen-moe-full-build')
WORK.mkdir(parents=True, exist_ok=True)
print('Output folder:', WORK)

In [ ]:
!git clone https://github.com/samanramezani1377-hub/Qwen-MoE-0.4B-.git /content/Qwen-MoE-0.4B-
%cd /content/Qwen-MoE-0.4B-
!pip -q install -U transformers accelerate safetensors datasets

In [ ]:
import torch, os, json, subprocess
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print('WARNING: no GPU detected. Full 24-layer run on CPU may take much longer.')

## گردآوری دادهٔ فارسی واقعی

این سلول ۲۰۰ سند فارسی واقعی از منابع مشخص‌شده در اسکریپت پروژه می‌گیرد. اگر منابع یا شبکه در دسترس نباشند، کار متوقف می‌شود و دادهٔ ساختگی جایگزین نمی‌شود. این حجم برای شروع و بررسی مسیر فنی است، نه آموزش نهایی.

In [ ]:
!python scripts/collect_persian_corpus.py --count 200 --min-chars 160 --max-chars 3000 --max-scan 30000 --output /content/fa_corpus.txt --manifest-output /content/fa_corpus.manifest.json
import shutil
shutil.copy2('/content/fa_corpus.txt', WORK / 'fa_corpus.txt')
shutil.copy2('/content/fa_corpus.manifest.json', WORK / 'fa_corpus.manifest.json')
print('Saved corpus and source/license manifest to Drive.')

## آموزش مرحله‌ای تمام ۲۴ لایه

هر سلول ۴ لایه را پردازش می‌کند و خروجی را مستقیماً روی Drive می‌نویسد. اگر نشست قطع شد، فقط سلول‌های مربوط به بازه‌های باقی‌مانده را اجرا کنید. تنظیم ثابت: ۳۲ متخصص، Top-2، دو دورهٔ آموزش اولیه، طول متن ۹۶ توکن. این تنظیم شروع فنی است؛ کیفیت نهایی نیاز به ارزیابی مستقل دارد.

In [ ]:
# Layer range 0-3
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "$WORK/checkpoints" --num-experts 32 --top-k 2 --epochs 2 --max-length 96 --batch-size 1 --train-batch-size 16 --device auto --start-layer 0 --end-layer 3
print('Finished range 0-3. Verify checkpoint files in Google Drive before continuing.')

In [ ]:
# Layer range 4-7
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "$WORK/checkpoints" --num-experts 32 --top-k 2 --epochs 2 --max-length 96 --batch-size 1 --train-batch-size 16 --device auto --start-layer 4 --end-layer 7
print('Finished range 4-7. Verify checkpoint files in Google Drive before continuing.')

In [ ]:
# Layer range 8-11
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "$WORK/checkpoints" --num-experts 32 --top-k 2 --epochs 2 --max-length 96 --batch-size 1 --train-batch-size 16 --device auto --start-layer 8 --end-layer 11
print('Finished range 8-11. Verify checkpoint files in Google Drive before continuing.')

In [ ]:
# Layer range 12-15
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "$WORK/checkpoints" --num-experts 32 --top-k 2 --epochs 2 --max-length 96 --batch-size 1 --train-batch-size 16 --device auto --start-layer 12 --end-layer 15
print('Finished range 12-15. Verify checkpoint files in Google Drive before continuing.')

In [ ]:
# Layer range 16-19
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "$WORK/checkpoints" --num-experts 32 --top-k 2 --epochs 2 --max-length 96 --batch-size 1 --train-batch-size 16 --device auto --start-layer 16 --end-layer 19
print('Finished range 16-19. Verify checkpoint files in Google Drive before continuing.')

In [ ]:
# Layer range 20-23
!python scripts/distill_all_qwen_ffn_layers.py --text-file /content/fa_corpus.txt --output-dir "$WORK/checkpoints" --num-experts 32 --top-k 2 --epochs 2 --max-length 96 --batch-size 1 --train-batch-size 16 --device auto --start-layer 20 --end-layer 23
print('Finished range 20-23. Verify checkpoint files in Google Drive before continuing.')

## بررسی فایل‌های تولیدشده

پس از اجرای همهٔ بازه‌ها، باید ۲۴ checkpoint داشته باشید. اگر تعداد کمتر است، بازه‌های مربوط به لایه‌های باقی‌مانده را اجرا کنید. این فایل‌ها هنوز مدل نهایی قابل بارگذاری یا GGUF نیستند.

In [ ]:
ckpt_dir = WORK / 'checkpoints'
files = sorted(ckpt_dir.glob('layer_*_top2_32experts.pt'))
print(f'Found {len(files)} / 24 layer checkpoints')
for f in files: print(f.name, f.stat().st_size, 'bytes')
missing = [i for i in range(24) if not (ckpt_dir / f'layer_{i:02d}_top2_32experts.pt').exists()]
print('Missing layers:', missing)
assert len(files) == 24, 'Not all layer checkpoints exist yet; run missing ranges first.'

## بعد از این مرحله

باید کلاس مدل سازگار با Transformers پیاده‌سازی شود تا همهٔ checkpointها را بارگذاری کند؛ سپس ذخیره/بارگذاری مجدد، inference کامل، آزمون‌های فارسی و استدلال، و بعد بررسی کوانتیزه‌سازی و runtime اندروید انجام می‌شود. این Notebook ادعا نمی‌کند که مدل کامل یا کیفیت/سرعت نهایی را ساخته یا تأیید کرده است.